### Importación y Preparación
Importamos las herramientas necesarias: `load_wine` para los datos, `SGDClassifier` para el modelo lineal, y `StandardScaler` para la normalización.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

### Carga y División del Dataset
Cargamos el dataset Wine, que contiene resultados de análisis químicos de vinos cultivados en la misma región de Italia por tres cultivadores diferentes.
Dividimos en conjuntos de entrenamiento (Train) y prueba (Test) para asegurar una evaluación honesta de la precisión.

In [ ]:
# Cargamos el dataset
data = load_wine()
X, y = data.data, data.target

# Dividimos en train y test para medir la precisión correctamente
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

### Datos Crudos (Sin Escalar)
Entrenamos el clasificador con los datos tal cual vienen. El algoritmo SGD (Descenso de Gradiente Estocástico) es sensible a la escala de las variables.
Medimos:
* **Iteraciones (`n_iter_`):** Cuántos pasos necesitó para encontrar el mínimo.
* **Precisión:** Qué tan bien clasifica.

In [ ]:
# Datos Crudos (Sin escalar)
print("Entrenando con datos CRUDOS")
clf_raw = SGDClassifier(max_iter=10000, tol=1e-3, random_state=42)
clf_raw.fit(X_train, y_train)

iters_raw = clf_raw.n_iter_
acc_raw = accuracy_score(y_test, clf_raw.predict(X_test))

print(f"Iteraciones para converger: {iters_raw}")
print(f"Precisión (Accuracy): {acc_raw:.4f}\n")

Entrenando con datos CRUDOS
Iteraciones para converger: 32
Precisión (Accuracy): 0.5556



### Datos Escalados (StandardScaler)
Repetimos el proceso pero normalizando los datos para que todas las características tengan media 0 y desviación estándar 1.

*Nota: Es crucial ajustar el scaler (`fit`) solo con los datos de entrenamiento y usar ese mismo ajuste para transformar los datos de prueba, evitando la fuga de datos.*

In [ ]:
# Datos Escalados (StandardScaler)
print("Entrenando con datos ESCALADOS")
# Aplicamos el escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf_scaled = SGDClassifier(max_iter=10000, tol=1e-3, random_state=42)
clf_scaled.fit(X_train_scaled, y_train)

iters_scaled = clf_scaled.n_iter_
acc_scaled = accuracy_score(y_test, clf_scaled.predict(X_test_scaled))

print(f"Iteraciones para converger: {iters_scaled}")
print(f"Precisión (Accuracy): {acc_scaled:.4f}")

Entrenando con datos ESCALADOS
Iteraciones para converger: 12
Precisión (Accuracy): 0.9815


### Comparativa de Resultados
Calculamos la diferencia de iteraciones entre ambos métodos. Se espera que con los datos escalados, el algoritmo converja mucho más rápido (menos iteraciones) y con mejor precisión.

In [ ]:
# Comparación
print(f"Diferencia de iteraciones: {iters_raw - iters_scaled}")

Diferencia de iteraciones: 20


### ¿Cómo cambia el número de pasos (iteraciones)?
El número de pasos se reduce drásticamente con los datos escalados.

Sin escalar: Es probable que el modelo necesite muchas iteraciones o incluso que alcance el límite máximo (max_iter) sin converger óptimamente. Esto sucede porque las características del vino tienen magnitudes muy diferentes (ej. Proline tiene valores de 1000, mientras que Nonflavanoid phenols son menores a 1). El "paisaje" de error se vuelve un valle muy alargado y estrecho, haciendo que el Descenso de Gradiente (SGD) rebote de un lado a otro (zig-zag) avanzando lentamente.

Con StandardScaler: El modelo suele converger en muy pocas iteraciones (frecuentemente menos de 20 o 30). Al escalar, el paisaje de error se vuelve más circular y uniforme, permitiendo que el algoritmo vaya directo al mínimo.

### ¿Afecta a la precisión?
Sí, y mucho.

El SGDClassifier es muy sensible a la escala de las características. Sin escalar, el gradiente estará dominado por la variable con mayor magnitud numérica (en este caso, Proline), ignorando otras variables importantes pero con valores pequeños.

Esto suele resultar en una precisión mucho más baja con los datos crudos. Al usar StandardScaler, todas las características contribuyen equitativamente a la decisión, mejorando significativamente la capacidad predictiva del modelo.